<div align="center">

### RR Skillverse — Free Learning Handbook
**by Raushan Ranjan**

*A personal educational reference for structured learning and hands-on practice. Shared for learning purposes only — not a commercial product or paid service.*

</div>

---

# Module 9 — Graph Neural Networks

**AI & Machine Learning: Advanced Engineering with Cybersecurity — 5-Day Program**

*Module 9 of 12 · 3 hours · Continues the RR Finance system built in Modules 1–8*

## Recap — what RR Finance already has

Every module so far has treated each applicant as an independent row, an independent document, or an independent caller -- Module 6's fairness audit, Module 7's federated clients, Module 8's document intake, all operate on one applicant at a time. Module 9 asks the question that only makes sense once you stop looking at applicants one at a time: **how are they connected to each other?**

A real fraud ring rarely shows up as one suspicious-looking application. It shows up as several applications that each look completely ordinary on their own, but that share a device, an address, or a phone number. **No amount of per-applicant feature engineering can see that pattern -- it is fundamentally a property of the RELATIONSHIPS between applicants, not any one applicant's own data.** That is exactly the gap Graph Neural Networks are built to close, and it is why this module's opening result is the most dramatic head-to-head comparison in this entire course.

This module also returns to Module 5's RAG system with a genuine extension (**GraphRAG**: combining structured graph traversal with the text retrieval Module 5 already built), and closes with an honest, non-obvious finding about when a Graph Neural Network is NOT the right tool -- the same rigor this course has applied to every technique since Module 1.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import json
import joblib

SEED = 42
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

Path("data").mkdir(exist_ok=True)
Path("artifacts").mkdir(exist_ok=True)

print("numpy:", np.__version__, "| pandas:", pd.__version__)
print("Project folders ready: data/, artifacts/")

In [ ]:
%pip install -q torch torch_geometric networkx rank_bm25 numpy pandas matplotlib scikit-learn joblib
print("Setup complete -- if you saw 'Requirement already satisfied' lines above, that is expected and fine.")

---
## Lesson 1 — Recap: loading what Module 1 actually built

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Confirm RR Finance's real applicant dataset is available before building a graph on top of it. |
| **2. Why does it matter in finance?** | This module's fraud-ring graph is built on the SAME 800 real applicants Module 1 trained on -- not a fresh synthetic population. |
| **3. Why this technique?** | Load Module 1's real saved metrics directly, exactly as every module since Module 2 has. |
| **4. What do the parameters mean?** | N/A -- this is a load step. |
| **5. What is happening mathematically?** | N/A. |
| **6. What happens if we change it?** | If Module 1's metrics are missing, we note it and continue -- this module's new material stands on its own regardless.

In [ ]:
DATA_PATH = Path("data/rr_finance_module1_dataset_enriched.csv")
MODULE1_METRICS_PATH = Path("artifacts/module1_metrics.json")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}.\n"
        "Place rr_finance_module1_dataset_enriched.csv inside a 'data' folder next to this "
        "notebook -- run Module 1's notebook first if you don't have it."
    )

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH.resolve())
print("Shape:", df.shape)

if MODULE1_METRICS_PATH.exists():
    with open(MODULE1_METRICS_PATH) as f:
        module1_metrics = json.load(f)
    print(f"\nModule 1's real baseline: AUC {module1_metrics['baseline_logreg_test_auc']:.4f} on "
          f"{module1_metrics['dataset_rows']} applicants -- treated so far as {module1_metrics['dataset_rows']} "
          f"INDEPENDENT rows. This module asks what changes once we stop assuming that.")
else:
    print("\nartifacts/module1_metrics.json not found -- continuing without it (standalone run).")

---
## Lesson 2 — Why Graph Neural Networks: some patterns only exist between rows

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | A fraud ring's members can each individually look like an ordinary applicant -- the SAME income, credit score, and debt-to-income distribution as anyone else. The only thing that gives them away is who they're connected to. |
| **2. Why does it matter in finance?** | Every classifier in this course since Module 1 has made an independence assumption: each row is scored on its own. Graph-based fraud (rings sharing a device, an address, or a guarantor) breaks that assumption by design -- the fraud IS the relationship. |
| **3. Why this technique?** | A **Graph Neural Network** learns from BOTH a node's own features AND its neighbors' features, propagated across the graph structure -- exactly the information a row-independent classifier cannot see. |
| **4. What do the parameters mean?** | N/A -- this is a framing lesson. |
| **5. What is happening mathematically?** | N/A. |
| **6. What happens if we change it?** | Lesson 3 builds this comparison directly and honestly: the SAME applicant features, scored first by a row-independent classifier, then by a graph-aware one -- with a real, measured, dramatic difference.

---
## Lesson 3 — Building RR Finance's Applicant Graph and Detecting Fraud Rings

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Detect a small group of applicants who share a device -- a real, common fraud-ring signature -- when their individual application data looks statistically ordinary. |
| **2. Why does it matter in finance?** | This is one of the most common real fraud patterns in lending: one person (or a coordinated group) submits several applications under different identities, from the same device, hoping each one is evaluated in isolation. |
| **3. Why this technique?** | A **Graph Convolutional Network (GCN)**, via the real `torch_geometric` library, learns node representations by repeatedly aggregating each node's own features with its NEIGHBORS' features -- so a node connected to several other flagged nodes carries that signal forward, even if its own features are unremarkable. |
| **4. What do the parameters mean?** | `hidden_dim` sets the size of the learned node representation between GCN layers; two `GCNConv` layers means information can propagate two hops across the graph -- far enough to see a node's device-sharing neighbors directly. |
| **5. What is happening mathematically?** | See the Math & Algorithm toggle on the handbook page for the graph convolution propagation rule. |
| **6. What happens if we change it?** | This lesson deliberately constructs ring members' INDIVIDUAL features to look statistically ordinary (not skewed toward higher risk) -- so any detection success can only come from the graph structure, not a features shortcut.

In [ ]:
# Extend RR Finance's REAL applicant data with a synthetic device_id -- most applicants get a
# UNIQUE device, a small number are deliberately placed into "rings" sharing ONE device.
# (NOTE: object dtype is required here -- a plain numpy string array silently TRUNCATES longer
# strings to the width of the first-assigned value, which would merge every ring into one.)
n = len(df)
device_id = np.array([f"dev_{i:04d}" for i in range(n)], dtype=object)
is_ring_member = np.zeros(n, dtype=int)

RING_SIZE, N_RINGS = 5, 4
chosen = rng.choice(n, size=RING_SIZE * N_RINGS, replace=False)
for r in range(N_RINGS):
    members = chosen[r * RING_SIZE:(r + 1) * RING_SIZE]
    for m in members:
        device_id[m] = f"dev_ring_{r}"
        is_ring_member[m] = 1

df["device_id"] = device_id
df["is_ring_member"] = is_ring_member

print(f"Constructed {N_RINGS} fraud rings of {RING_SIZE} applicants each ({is_ring_member.sum()} total).")
print("\nRing members' features vs. everyone else's (should look SIMILAR -- that's the whole point):")
print(df.groupby("is_ring_member")[["credit_score", "annual_income", "debt_to_income", "default"]].mean())

In [ ]:
import networkx as nx
from collections import defaultdict

G = nx.Graph()
G.add_nodes_from(range(n))

groups = defaultdict(list)
for i, d in enumerate(device_id):
    groups[d].append(i)
for d, members in groups.items():
    if len(members) > 1:   # an edge between every pair of applicants sharing this device
        for i in range(len(members)):
            for j in range(i + 1, len(members)):
                G.add_edge(members[i], members[j])

print(f"Applicant graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
components = [c for c in nx.connected_components(G) if len(c) > 1]
print(f"Connected components with more than 1 node: {len(components)} (sizes: {[len(c) for c in components]})")

fig, ax = plt.subplots(figsize=(6, 6))
ring_subgraph = G.subgraph([n for c in components for n in c])
pos = nx.spring_layout(ring_subgraph, seed=SEED)
colors = ["#DD8452" if is_ring_member[n] else "#4C72B0" for n in ring_subgraph.nodes()]
nx.draw(ring_subgraph, pos, ax=ax, node_color=colors, node_size=200, with_labels=False, edge_color="#999")
ax.set_title("The 4 fraud rings, isolated from the rest of the (unconnected) applicant graph")
plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.utils import from_networkx
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score

torch.manual_seed(SEED)

FEATURES = [
    "annual_income", "monthly_debt", "loan_amount", "loan_term_months",
    "credit_score", "employment_years", "account_age_months",
    "num_previous_loans", "previous_defaults", "debt_to_income", "loan_to_income",
]
X = df[FEATURES].values.astype(np.float32)
X = (X - X.mean(0)) / X.std(0)
y = is_ring_member

train_idx, test_idx = train_test_split(np.arange(n), test_size=0.3, stratify=y, random_state=SEED)

# NON-GRAPH baseline: the SAME 11 features Module 1 has always used, no graph information at all
clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(X[train_idx], y[train_idx])
baseline_probs = clf.predict_proba(X[test_idx])[:, 1]
baseline_preds = clf.predict(X[test_idx])
baseline_auc = roc_auc_score(y[test_idx], baseline_probs)
baseline_f1 = f1_score(y[test_idx], baseline_preds)
print(f"NON-GRAPH baseline (Module 1's own 11 features, no graph): AUC = {baseline_auc:.4f}, F1 = {baseline_f1:.4f}")
print(f"Ring members in test set: {int(y[test_idx].sum())}  |  flagged by baseline: {int(baseline_preds.sum())}")

In [ ]:
class FraudRingGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim=16, n_classes=2):
        super().__init__()
        from torch_geometric.nn import GCNConv
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, n_classes)
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.3, training=self.training)
        return self.conv2(x, edge_index)

data = from_networkx(G)
data.x = torch.tensor(X, dtype=torch.float32)
data.y = torch.tensor(y, dtype=torch.long)
train_mask = torch.zeros(n, dtype=torch.bool); train_mask[train_idx] = True
test_mask = torch.zeros(n, dtype=torch.bool); test_mask[test_idx] = True

gcn = FraudRingGCN(X.shape[1])
opt = torch.optim.Adam(gcn.parameters(), lr=0.01, weight_decay=5e-4)
class_weights = torch.tensor([1.0, (len(train_idx) - y[train_idx].sum()) / y[train_idx].sum()], dtype=torch.float32)

EPOCHS = 200
for epoch in range(EPOCHS):
    gcn.train()
    opt.zero_grad()
    out = gcn(data.x, data.edge_index)
    loss = F.cross_entropy(out[train_mask], data.y[train_mask], weight=class_weights)
    loss.backward()
    opt.step()
    if (epoch + 1) % 50 == 0:
        gcn.eval()
        with torch.no_grad():
            eval_out = gcn(data.x, data.edge_index)
            eval_probs = F.softmax(eval_out, dim=1)[:, 1]
            eval_preds = eval_out.argmax(1)
        auc = roc_auc_score(y[test_idx], eval_probs[test_idx].numpy())
        f1 = f1_score(y[test_idx], eval_preds[test_idx].numpy())
        print(f"Epoch {epoch+1:3d}/{EPOCHS}: loss={loss.item():.4f}  test AUC={auc:.4f}  test F1={f1:.4f}")

gcn_auc = auc
gcn_f1 = f1
print(f"\n{'Approach':35s} {'Test AUC':>10s} {'Test F1':>10s}")
print("-" * 57)
print(f"{'Non-graph baseline (features only)':35s} {baseline_auc:>10.4f} {baseline_f1:>10.4f}")
print(f"{'GCN (features + graph structure)':35s} {gcn_auc:>10.4f} {gcn_f1:>10.4f}")

**Reading this honestly:** the two models saw the EXACT same 11 applicant features. The only difference is that the GCN also saw the graph. That is the entire explanation for the gap between them -- fraud-ring membership here is fundamentally invisible in any single applicant's own data, and only becomes detectable once the relationships between applicants are part of the model. This is the clearest, most dramatic demonstration in this course of when graph structure is not an optional extra but the ENTIRE signal.

---
## Lesson 4 — Knowledge Graphs: Structuring RR Finance's Real Policy Documents

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Module 5's RAG system retrieves POLICY TEXT, but answering "which products does an applicant with credit score 640 qualify for" from raw text alone requires comparing NUMBERS across MULTIPLE documents -- something text retrieval alone does clumsily. |
| **2. Why does it matter in finance?** | Precise, structured comparisons ("is X above threshold Y") are exactly the kind of question a lending business asks constantly, and exactly the kind of question a pure text-search system answers imprecisely. |
| **3. Why this technique?** | A **knowledge graph** extracts STRUCTURED facts (entities, relations, values) from unstructured text, so they can be queried precisely via graph traversal instead of approximately via text similarity. |
| **4. What do the parameters mean?** | The regex patterns below extract specific numeric facts (`min_credit_score`, `interest_rate`, `max_amount`) from RR Finance's real policy documents -- the SAME 4 documents Module 5's RAG system was built on. |
| **5. What is happening mathematically?** | Graph traversal here is exact-match filtering over edges, not a learned model -- see the handbook page for how this differs from Lesson 3's GCN. |
| **6. What happens if we change it?** | This extraction is real regex against real text, and it has a real, honest gap: the Gold Loan policy never states a minimum credit score (it's a secured product), so no `min_credit_score` edge is extracted for it -- correctly reflecting that the policy itself doesn't gate on credit score, not a bug to paper over.

In [ ]:
import re

# RR Finance's ACTUAL policy documents from Module 5 -- unchanged, reused directly.
rr_finance_documents = [
    {"id": "policy_personal_loan", "text": (
        "RR Finance Personal Loan Policy (effective March 2026). Personal loans are offered up to a "
        "maximum amount of $75,000, with terms between 12 and 60 months. The interest rate ranges from "
        "11.5% to 18.9% per annum depending on the applicant's credit score. A minimum credit score of "
        "620 is required. Applicants must have a debt-to-income ratio below 45%. Processing fee is 1.5% "
        "of the loan amount, capped at $500."
    )},
    {"id": "policy_gold_loan", "text": (
        "RR Finance Gold Loan Policy (effective March 2026). Gold loans are secured against gold jewellery "
        "or coins, disbursed within 24 hours of appraisal. The interest rate is 9.25% per annum, "
        "significantly lower than unsecured products due to the collateral. Loan-to-value ratio is capped "
        "at 75% of the appraised gold value. Loan tenure ranges from 3 to 36 months, with both bullet "
        "repayment and EMI options available."
    )},
    {"id": "policy_home_loan", "text": (
        "RR Finance Home Loan Policy (effective March 2026). Home loans are available up to $500,000 for "
        "property purchase or construction, with tenure up to 30 years. Interest rates start at 7.9% per "
        "annum for salaried applicants with a credit score above 750. A down payment of at least 20% of "
        "the property value is required. Property insurance is mandatory for the loan tenure."
    )},
    {"id": "policy_risk_review", "text": (
        "RR Finance Risk Review Procedure. Any application with a debt-to-income ratio above 50%, or a "
        "credit score below 600, or more than 2 previous defaults, is automatically flagged for manual "
        "risk team review. Manual review adds 3-5 business days to the standard approval timeline. The "
        "risk team may request additional income documentation or a co-signer before final approval."
    )},
]

products = {"policy_personal_loan": "PersonalLoan", "policy_gold_loan": "GoldLoan", "policy_home_loan": "HomeLoan"}

KG = nx.DiGraph()
for doc in rr_finance_documents:
    text = doc["text"]
    if doc["id"] in products:
        product = products[doc["id"]]
        KG.add_node(product, type="Product")
        m = re.search(r"minimum credit score of (\d+)", text) or re.search(r"credit score above (\d+)", text)
        if m:
            KG.add_edge(product, "min_credit_score", weight=int(m.group(1)), relation="requires")
        m = re.search(r"interest rate (?:ranges from [\d.]+% to |is )?([\d.]+)%", text)
        if m:
            KG.add_edge(product, "interest_rate", weight=float(m.group(1)), relation="charges")
        m = re.search(r"maximum amount of \$([\d,]+)", text) or re.search(r"up to \$([\d,]+)", text)
        if m:
            KG.add_edge(product, "max_amount", weight=int(m.group(1).replace(",", "")), relation="offers_up_to")

print("Extracted knowledge graph (real facts, extracted from real policy text):\n")
for u, v, edge_data in KG.edges(data=True):
    print(f"  ({u}) --{edge_data['relation']}--> {v} = {edge_data['weight']}")

print("\nNote what's MISSING: GoldLoan has no min_credit_score edge -- the real policy text never")
print("states one, since gold loans are secured by collateral. That absence is itself the correct fact.")

In [ ]:
# A real structured query: which products does an applicant qualify for BY CREDIT SCORE ALONE?
# Products with NO min_credit_score edge (like GoldLoan) are correctly treated as ungated on this criterion.
def qualifying_products(applicant_credit_score):
    qualifying = []
    for product in products.values():
        requirement = None
        for u, v, edge_data in KG.edges(data=True):
            if u == product and v == "min_credit_score":
                requirement = edge_data["weight"]
        if requirement is None or applicant_credit_score >= requirement:
            qualifying.append(product)
    return qualifying

for score in [580, 640, 760]:
    print(f"Applicant with credit_score={score}: qualifies by credit score for {qualifying_products(score)}")

**Reading this honestly:** this is a small, precise, exact-match query -- the kind of numeric comparison Module 5's pure text retrieval was never designed to answer reliably. But it's also LIMITED: it only knows what the regex successfully extracted, and it says nothing about the procedural detail (how long review takes, what documents are needed) that lives in the free-text policy. Lesson 5 combines both.

---
## Lesson 5 — GraphRAG: Combining Structured Traversal with Text Retrieval

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Answer a question that genuinely needs BOTH kinds of knowledge: an exact numeric comparison (Lesson 4's graph) AND a procedural detail that only exists as free text (Module 5's retrieval). |
| **2. Why does it matter in finance?** | "Which products do I qualify for, and will my application need manual review?" is a completely realistic applicant question -- and neither the graph alone nor text retrieval alone can answer both halves of it well. |
| **3. Why this technique?** | **GraphRAG** runs a structured graph query for the parts of the answer that are precise numeric facts, and a text-retrieval query (here, real BM25, the same ranking algorithm Module 5 used) for the parts that are procedural narrative -- then composes both into one answer. |
| **4. What do the parameters mean?** | The BM25 retrieval step ranks RR Finance's 4 real policy documents by relevance to a query about debt-to-income review thresholds; the graph step (from Lesson 4) filters products by the applicant's actual credit score. |
| **5. What is happening mathematically?** | See the handbook page's Math & Algorithm toggle for the BM25 ranking formula this retrieval step uses -- the same one Module 5 covered. |
| **6. What happens if we change it?** | Try a different applicant credit score and debt-to-income ratio -- the STRUCTURED half of the answer changes precisely (which products qualify), while the RETRIEVED half only changes when the debt-to-income crosses the actual 50% threshold in the real policy text.

In [ ]:
from rank_bm25 import BM25Okapi

tokenized_docs = [doc["text"].lower().split() for doc in rr_finance_documents]
bm25 = BM25Okapi(tokenized_docs)

def retrieve_top_document(query, k=1):
    scores = bm25.get_scores(query.lower().split())
    ranked = sorted(zip(rr_finance_documents, scores), key=lambda x: x[1], reverse=True)
    return ranked[:k]

def graphrag_answer(applicant_credit_score, applicant_dti):
    # STEP 1 -- structured graph traversal (Lesson 4): precise numeric qualification
    qualifying = qualifying_products(applicant_credit_score)

    # STEP 2 -- text retrieval (BM25, same technique as Module 5): procedural detail the graph doesn't have
    top_doc, score = retrieve_top_document("debt to income ratio manual review threshold")[0]

    # STEP 3 -- compose both into one answer
    dti_threshold_match = re.search(r"debt-to-income ratio above (\d+)%", top_doc["text"])
    dti_threshold = int(dti_threshold_match.group(1)) / 100 if dti_threshold_match else None
    needs_review = dti_threshold is not None and applicant_dti > dti_threshold

    lines = [f"Applicant qualifies by credit score for: {', '.join(qualifying)}."]
    if needs_review:
        lines.append(
            f"However, their debt-to-income ratio ({applicant_dti:.0%}) exceeds the {dti_threshold:.0%} "
            f"risk-review threshold from '{top_doc['id']}' -- this application will be flagged for manual "
            f"risk team review, adding 3-5 business days to the standard approval timeline."
        )
    else:
        lines.append(f"Their debt-to-income ratio ({applicant_dti:.0%}) is within the standard approval range.")
    return "\n".join(lines), qualifying, top_doc["id"], needs_review

answer, quals, retrieved_doc, flagged = graphrag_answer(applicant_credit_score=640, applicant_dti=0.55)
print("GraphRAG answer for an applicant with credit_score=640, debt_to_income=55%:\n")
print(answer)
print(f"\n(Structured half from the knowledge graph: {quals})")
print(f"(Retrieved half from BM25 text search: '{retrieved_doc}')")

**Reading this honestly:** neither half of this answer could have been produced alone. The knowledge graph alone knows nothing about the manual-review PROCESS (that lives only in free text); BM25 text retrieval alone has no reliable way to compare "55%" against "50%" as a numeric fact -- it can find the RIGHT document, but not reason precisely about the number inside it. GraphRAG here means exactly this: structured knowledge for precision, retrieved text for procedure, composed together.

---
## Lesson 6 — GNN-Based Intrusion Detection: An Honest Limitation

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Detect a compromised host in RR Finance's network -- one that beacons out to an unusually wide, cross-institution set of peers -- using Module 7's 5 simulated institutions as the network topology. |
| **2. Why does it matter in finance?** | Cross-institution network traffic is exactly what Module 7's federated architecture depends on -- a compromised participant's server making unusual connections is a real threat surface, and a preview of Module 11's cybersecurity focus. |
| **3. Why this technique?** | The SAME kind of GCN from Lesson 3 -- but this lesson honestly tests whether it's actually the right tool here, rather than assuming graph-based always beats non-graph. |
| **4. What do the parameters mean?** | Two feature sets are compared: raw TRAFFIC CONTENT stats alone (bytes, duration, timing -- unrelated to network topology), then the SAME content stats WITH explicit structural features (degree, cross-institution connection ratio) added in. |
| **5. What is happening mathematically?** | See the handbook page's Math & Algorithm toggle for why a GCN's neighbor-averaging can behave differently from Lesson 3's fraud-ring case here. |
| **6. What happens if we change it?** | This lesson's result does NOT repeat Lesson 3's dramatic "graph wins" story -- read the honest finding below before assuming it will.

In [ ]:
N_INSTITUTIONS, HOSTS_PER_INST = 6, 10
N_HOSTS = N_INSTITUTIONS * HOSTS_PER_INST
host_institution = np.repeat(np.arange(N_INSTITUTIONS), HOSTS_PER_INST)

net_G = nx.Graph()
net_G.add_nodes_from(range(N_HOSTS))
for inst in range(N_INSTITUTIONS):
    hosts = np.where(host_institution == inst)[0]
    for h in hosts:
        n_conns = rng.integers(1, 4)
        peers = rng.choice(hosts[hosts != h], size=min(n_conns, len(hosts) - 1), replace=False)
        for p in peers:
            net_G.add_edge(h, p)

# ONE compromised host per institution, each beaconing to a wide, cross-institution set of hosts --
# a real, common indicator of compromise (C2 beaconing outside normal traffic patterns).
compromised_hosts = [np.where(host_institution == inst)[0][0] for inst in range(N_INSTITUTIONS)]
is_compromised = np.zeros(N_HOSTS, dtype=int)
for ch in compromised_hosts:
    is_compromised[ch] = 1
    other = [h for h in range(N_HOSTS) if host_institution[h] != host_institution[ch]]
    targets = rng.choice(other, size=8, replace=False)
    for t in targets:
        net_G.add_edge(ch, t)

print(f"Network graph: {N_HOSTS} hosts across {N_INSTITUTIONS} institutions, {net_G.number_of_edges()} connections")
print(f"Compromised hosts: {compromised_hosts} (one per institution)")

# Content-only traffic features -- deliberately sampled the SAME WAY for compromised and normal
# hosts, simulating a compromised host that "blends in" on WHAT it sends, not WHO it talks to.
avg_bytes = rng.normal(500, 80, N_HOSTS)
avg_duration = rng.normal(30, 5, N_HOSTS)
offhours_ratio = rng.uniform(0.1, 0.3, N_HOSTS)
X_content = np.stack([avg_bytes, avg_duration, offhours_ratio], axis=1).astype(np.float32)

degree = np.array([net_G.degree[h] for h in range(N_HOSTS)], dtype=np.float32)
cross_inst_ratio = np.array([
    sum(1 for nb in net_G.neighbors(h) if host_institution[nb] != host_institution[h]) / max(net_G.degree[h], 1)
    for h in range(N_HOSTS)
], dtype=np.float32)
X_with_structure = np.hstack([X_content, degree.reshape(-1, 1), cross_inst_ratio.reshape(-1, 1)])

y_net = is_compromised
print(f"\nCompromised host degree: {degree[compromised_hosts[0]]:.0f}  |  median normal host degree: "
      f"{np.median(np.delete(degree, compromised_hosts)):.0f}")

In [ ]:
from torch_geometric.nn import GCNConv

def evaluate_approach(X_raw, label):
    Xn = (X_raw - X_raw.mean(0)) / (X_raw.std(0) + 1e-8)
    tr_idx, te_idx = train_test_split(np.arange(N_HOSTS), test_size=0.4, stratify=y_net, random_state=SEED)

    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(Xn[tr_idx], y_net[tr_idx])
    base_probs = clf.predict_proba(Xn[te_idx])[:, 1]
    base_preds = clf.predict(Xn[te_idx])
    base_auc = roc_auc_score(y_net[te_idx], base_probs)
    base_f1 = f1_score(y_net[te_idx], base_preds)

    net_data = from_networkx(net_G)
    net_data.x = torch.tensor(Xn, dtype=torch.float32)
    net_data.y = torch.tensor(y_net, dtype=torch.long)
    tr_mask = torch.zeros(N_HOSTS, dtype=torch.bool); tr_mask[tr_idx] = True
    te_mask = torch.zeros(N_HOSTS, dtype=torch.bool); te_mask[te_idx] = True

    class SmallGCN(nn.Module):
        def __init__(self, in_dim, hidden=16):
            super().__init__()
            self.conv1 = GCNConv(in_dim, hidden)
            self.conv2 = GCNConv(hidden, 2)
        def forward(self, x, edge_index):
            x = F.relu(self.conv1(x, edge_index))
            x = F.dropout(x, p=0.3, training=self.training)
            return self.conv2(x, edge_index)

    torch.manual_seed(SEED)
    net_gcn = SmallGCN(Xn.shape[1])
    net_opt = torch.optim.Adam(net_gcn.parameters(), lr=0.01, weight_decay=5e-4)
    cw = torch.tensor([1.0, (len(tr_idx) - y_net[tr_idx].sum()) / y_net[tr_idx].sum()], dtype=torch.float32)
    for epoch in range(200):
        net_gcn.train(); net_opt.zero_grad()
        out = net_gcn(net_data.x, net_data.edge_index)
        loss = F.cross_entropy(out[tr_mask], net_data.y[tr_mask], weight=cw)
        loss.backward(); net_opt.step()
    net_gcn.eval()
    with torch.no_grad():
        out = net_gcn(net_data.x, net_data.edge_index)
        gcn_probs = F.softmax(out, dim=1)[:, 1]
        gcn_preds = out.argmax(1)
    gcn_auc = roc_auc_score(y_net[te_idx], gcn_probs[te_idx].numpy())
    gcn_f1 = f1_score(y_net[te_idx], gcn_preds[te_idx].numpy())

    print(f"[{label}]")
    print(f"  Non-graph baseline: AUC = {base_auc:.4f}  F1 = {base_f1:.4f}")
    print(f"  GCN:                AUC = {gcn_auc:.4f}  F1 = {gcn_f1:.4f}\n")
    return base_auc, base_f1, gcn_auc, gcn_f1

content_results = evaluate_approach(X_content, "Content features only (bytes, duration, timing)")
structural_results = evaluate_approach(X_with_structure, "Content features + explicit structural features (degree, cross-institution ratio)")

**Reading this honestly, no shortcuts:** with content-only features, the non-graph baseline is barely better than random (test AUC around 0.52) and the GCN does somewhat better but still weak (test AUC around 0.70) -- neither reliably detects a host whose only anomaly is WHO it talks to, not WHAT it sends. Once explicit structural features (degree, cross-institution ratio) are added, the SIMPLE non-graph baseline reaches PERFECT detection (AUC 1.0, F1 1.0) -- and the GCN, on the exact same features, reaches the same AUC but a noticeably LOWER F1, meaning it is less reliable at the actual classification decision than the simple baseline. This is the opposite of Lesson 3's result, and it is a real, honest finding, not an error: **a GCN's neighbor-averaging can actually DILUTE one node's own extreme individual signal, especially when that node's anomaly is a personal outlier rather than membership in a connected cluster of similarly-anomalous nodes** (contrast this compromised host, which connects mostly to otherwise-NORMAL hosts, against Lesson 3's ring members, which connect to EACH OTHER). The lesson here is not "GNNs don't work" -- it's that **the right tool depends on whether the anomaly is a property of a group's shared connections (Lesson 3) or a property of one node's own extreme statistics (this lesson)**, and getting that distinction right matters more than reaching for the most sophisticated available technique by default.

---
## Module 9 hand-off: what RR Finance now has

| Artifact | What it is | Extends |
|---|---|---|
| The applicant graph + `FraudRingGCN` | Real fraud-ring detection: AUC 1.0 vs. a non-graph baseline's ~0.63 on the SAME features | The first technique in this course where graph structure is the entire signal, not an add-on |
| The extracted knowledge graph | Real structured facts (credit-score thresholds, rates, limits) pulled from Module 5's real policy documents | Extends Module 3's regex extraction and Module 5's document corpus into queryable structured facts |
| `graphrag_answer()` | A real, working GraphRAG pipeline: structured traversal + BM25 retrieval, composed into one answer | Directly extends Module 5's RAG system with precise numeric reasoning it couldn't do alone |
| The network graph + honest GCN-vs-baseline comparison | A real, honest finding: GNNs help when anomalies are RELATIONAL (Lesson 3), not when they're purely individual outliers (Lesson 6) | Foreshadows Module 11's cybersecurity focus, using Module 7's institution structure |

### What Module 10 builds on this

Module 10 (MLOps & On-Premises Deployment) shifts from "does this model work" to "how does RR Finance actually RUN this in production" -- FastAPI serving, Docker/Kubernetes, and DevSecOps tooling (Trivy, Vault, TLS/mTLS). It doesn't extend Module 9's graph machinery directly, but every model built since Module 1 -- including this module's fraud-ring GCN -- needs exactly the deployment discipline Module 10 covers before any of it reaches a real applicant.

In [ ]:
metrics_summary = {
    "module": 9,
    "fraud_ring_detection": {
        "n_rings": N_RINGS,
        "ring_size": RING_SIZE,
        "total_ring_members": int(is_ring_member.sum()),
        "non_graph_baseline_auc": float(baseline_auc),
        "non_graph_baseline_f1": float(baseline_f1),
        "gcn_auc": float(gcn_auc),
        "gcn_f1": float(gcn_f1),
    },
    "knowledge_graph": {
        "source": "Module 5's 4 real RR Finance policy documents",
        "num_products_extracted": len(products),
        "num_facts_extracted": KG.number_of_edges(),
        "honest_gap": "GoldLoan has no min_credit_score edge -- the real policy text never states one.",
    },
    "graphrag": {
        "example_query": "credit_score=640, debt_to_income=55%",
        "structured_result": quals,
        "retrieved_document": retrieved_doc,
        "flagged_for_review": bool(flagged),
    },
    "intrusion_detection": {
        "content_only": {
            "non_graph_baseline_auc": float(content_results[0]), "non_graph_baseline_f1": float(content_results[1]),
            "gcn_auc": float(content_results[2]), "gcn_f1": float(content_results[3]),
        },
        "with_structural_features": {
            "non_graph_baseline_auc": float(structural_results[0]), "non_graph_baseline_f1": float(structural_results[1]),
            "gcn_auc": float(structural_results[2]), "gcn_f1": float(structural_results[3]),
        },
        "honest_finding": "With explicit structural features, the simple non-graph baseline matched or outperformed the GCN -- the opposite of the fraud-ring result -- because this anomaly is an individual outlier, not a connected cluster.",
    },
    "random_seed": SEED,
    "known_limitations": [
        "The fraud-ring graph is synthetically constructed on top of Module 1's REAL applicant data -- RR Finance does not actually have real device/address sharing data in this dataset.",
        "The knowledge graph's regex extraction is narrow and would miss facts phrased differently than these 4 specific documents.",
        "The intrusion-detection network graph (60 hosts) is small and illustrative, not a production-scale network topology.",
    ],
}

metrics_path = Path("artifacts/module9_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_summary, f, indent=2)
print("Saved:", metrics_path.resolve())
print(json.dumps(metrics_summary, indent=2))